In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# Pathsクラス
class Paths:
    p = "/Users/shirokoshikentaro/Desktop/Python3/house-prices-advanced-regression-techniques"
    train = p + "/train.csv"
    test = p + "/test.csv"
    sample = p + "/sample_submission.csv"

# === データ読み込み ===
train_raw = pd.read_csv(Paths.train)
test_raw = pd.read_csv(Paths.test)

# === 新規特徴量の作成 ===
print("新規特徴量を作成中...")

train_raw["QualGrLiv"] = train_raw["OverallQual"] * train_raw["GrLivArea"]
test_raw["QualGrLiv"] = test_raw["OverallQual"] * test_raw["GrLivArea"]

train_raw["TotalSF"] = train_raw["TotalBsmtSF"] + train_raw["1stFlrSF"] + train_raw["2ndFlrSF"]
test_raw["TotalSF"] = test_raw["TotalBsmtSF"] + test_raw["1stFlrSF"] + test_raw["2ndFlrSF"]

train_raw["QualTotalSF"] = train_raw["OverallQual"] * train_raw["TotalSF"]
test_raw["QualTotalSF"] = test_raw["OverallQual"] * test_raw["TotalSF"]

train_raw["Age"] = train_raw["YrSold"] - train_raw["YearBuilt"]
test_raw["Age"] = test_raw["YrSold"] - test_raw["YearBuilt"]

train_raw["RemodAge"] = train_raw["YrSold"] - train_raw["YearRemodAdd"]
test_raw["RemodAge"] = test_raw["YrSold"] - test_raw["YearRemodAdd"]

train_raw["TotalBath"] = (train_raw["FullBath"] + 0.5*train_raw["HalfBath"].fillna(0) + 
                          train_raw["BsmtFullBath"].fillna(0) + 0.5*train_raw["BsmtHalfBath"].fillna(0))
test_raw["TotalBath"] = (test_raw["FullBath"] + 0.5*test_raw["HalfBath"].fillna(0) + 
                         test_raw["BsmtFullBath"].fillna(0) + 0.5*test_raw["BsmtHalfBath"].fillna(0))

train_raw["AreaPerRoom"] = train_raw["GrLivArea"] / train_raw["TotRmsAbvGrd"].replace(0, 1)
test_raw["AreaPerRoom"] = test_raw["GrLivArea"] / test_raw["TotRmsAbvGrd"].replace(0, 1)

train_raw["GarageScore"] = train_raw["GarageCars"] * train_raw["GarageArea"]
test_raw["GarageScore"] = test_raw["GarageCars"] * test_raw["GarageArea"]

# === 特徴量定義 ===
numeric_features = [
    "LotArea", "YearBuilt", "YearRemodAdd",
    "GrLivArea", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF",
    "FullBath", "BedroomAbvGr", "TotRmsAbvGrd",
    "GarageCars", "GarageArea",
    "OverallQual", "OverallCond",
    "QualGrLiv", "TotalSF", "QualTotalSF", 
    "Age", "RemodAge", "TotalBath", 
    "AreaPerRoom", "GarageScore"
]

categorical_features = [
    "Neighborhood", "BldgType", "HouseStyle",
    "MSZoning", "Foundation", "GarageType"
]

features = numeric_features + categorical_features
print(f"使用する特徴量: {len(features)}個")

# === 特徴量を抽出 ===
train_features = train_raw[features].copy()
test_features = test_raw[features].copy()

# === 欠損値処理 ===
for col in numeric_features:
    train_features[col] = train_features[col].fillna(0)
    test_features[col] = test_features[col].fillna(0)

for col in categorical_features:
    train_features[col] = train_features[col].fillna("None")
    test_features[col] = test_features[col].fillna("None")

# === エンコーディング ===
train_encoded = {}
test_encoded = {}

for col in numeric_features:
    train_encoded[col] = train_features[col].values
    test_encoded[col] = test_features[col].values

for col in categorical_features:
    all_data = pd.concat([train_features[col].astype(str), test_features[col].astype(str)], axis=0)
    le = LabelEncoder()
    le.fit(all_data)
    train_encoded[col] = le.transform(train_features[col].astype(str))
    test_encoded[col] = le.transform(test_features[col].astype(str))

# === DataFrame作成 ===
X_train = pd.DataFrame(train_encoded, columns=features)
X_test = pd.DataFrame(test_encoded, columns=features)

# ★★★ 対数変換 ★★★
y_train = np.log1p(train_raw["SalePrice"])

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train (log) - 平均: {y_train.mean():.3f}")

# === モデルパラメータ ===
params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 16,
    "learning_rate": 0.1,
    "n_estimators": 10000,
    "random_state": 123,
    "verbose": -1,
}

# === CV実行 ===
n_splits = 5
cv = KFold(n_splits=n_splits, shuffle=True, random_state=123)
metrics = []

for nfold, (train_index, valid_index) in enumerate(cv.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_index], y_train.iloc[train_index]
    X_val, y_val = X_train.iloc[valid_index], y_train.iloc[valid_index]

    model_fold = lgb.LGBMRegressor(**params)
    model_fold.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])

    y_val_pred = model_fold.predict(X_val)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    metrics.append(val_rmse)
    print(f"Fold {nfold}: {val_rmse:.5f}")

print(f"\n[CV] {np.mean(metrics):.5f}±{np.std(metrics):.5f}")

# === 全データで学習 ===
model = lgb.LGBMRegressor(**params)
model.fit(X_train, y_train, eval_set=[(X_train, y_train)], callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)])

# === 予測 ===
y_test_pred_log = model.predict(X_test)
y_test_pred = np.expm1(y_test_pred_log)  # ★元のスケールに戻す

print(f"\n予測価格: ${y_test_pred.mean():,.0f}")

# === 提出ファイル ===
df_submit = pd.DataFrame({"Id": test_raw["Id"], "SalePrice": y_test_pred})
df_submit.to_csv("submission_log_with_features.csv", index=False)
print("✅ 保存完了！")

新規特徴量を作成中...
使用する特徴量: 28個

X_train shape: (1460, 28)
y_train (log) - 平均: 12.024
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[68]	valid_0's rmse: 0.117659
Fold 0: 0.11766
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[208]	valid_0's rmse: 0.138327
Fold 1: 0.13833
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[76]	valid_0's rmse: 0.135546
Fold 2: 0.13555
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[143]	valid_0's rmse: 0.138104
Fold 3: 0.13810
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[308]	valid_0's rmse: 0.13177
Fold 4: 0.13177

[CV] 0.13228±0.00768
[100]	training's rmse: 0.0757703
[200]	training's rmse: 0.0568496
[300]	training's rmse: 0.0440439
[400]	training's rmse: 0.0353071
[500]	training's rmse: 0.028677
[600]	training's 